<a href="https://colab.research.google.com/github/yse-eds-cert/yse-eds-cert-classroom-code-along-notebooks-code_along_notebooks/blob/main/course-4-environmental-analysis/module-13/C4M13L2-text-analysis-for-env-data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 2: Text Analysis for Environmental Data

This notebook introduces fundamental concepts and techniques for analyzing environmental text data, focusing on preprocessing and dictionary-based methods for exploration.

## Learning Objectives:
- Load and clean environmental text data (policy documents, NGO statements, social media)
- Perform tokenization and stopword removal
- Conduct word frequency analysis
- Apply dictionary-based methods (sentiment, environment-related dictionaries)
- Create visualizations: word clouds, time series of keyword usage

Let's get started by loading the necessary packages!


In [ ]:
# run this line of code to quickly install the necessary spatial packages
system('apt-get install r-cran-knitr r-cran-scales')

In [ ]:
# Load the required packages
install.packages("tidytext")
install.packages("wordcloud2")
install.packages("tm")
install.packages("SnowballC")
library(tidytext)     # Text mining with tidy data principles
library(wordcloud2)   # Word cloud visualization
library(tm)          # Text mining utilities
library(SnowballC)   # Stemming
library(ggplot2)     # Data visualization
library(dplyr)       # Data manipulation
library(stringr)     # String manipulation
library(knitr)       # For nice table output
library(scales)      # For better plot scales
library(readr)       # For reading in csvs
library(tidyr)       # For data cleaning


## Part 1: Loading and Cleaning Environmental Text Data

### Understanding Our Dataset

We'll be working with a climate topics dataset that contains social media posts and news articles related to climate change. This type of data is common in environmental research and policy analysis.

The dataset contains:

- **text**: The actual text content
- **label**: Topic classification (0-4, representing different climate-related topics)
  - **0**: General climate discussion - References of people discussing and spreading awareness about climate change without a specific focus, and references of climate change affecting suburban lives
  - **1**: Politics and policy - Quotes from world leaders on climate change, references to institutional actions (like UN initiatives), and policies such as New Green Deal and COP25
  - **2**: Ocean/Water - Effects of climate change on ocean and water body biodiversity, water-based activities that accelerate climate change, and land biodiversity adaptation to climate effects
  - **3**: Agriculture/forestry - Effects of climate change on land biodiversity and crop yields, activities like deforestation and fossil fuel burning, and land biodiversity adaptation
  - **4**: Climate disasters and impacts - Natural disasters influenced by climate change including wildfires, floods, hurricanes, and droughts, with opinions and information about specific disaster instances

Let's start by loading and exploring our data:



In [ ]:
# Load the climate topics dataset
climate_data <- read_csv("https://github.com/yse-eds-cert/yse-eds-cert-classroom-code-along-notebooks-code_along_notebooks/raw/refs/heads/main/course-4-environmental-analysis/module-13/data/climate_data_course.csv")

# Examine the structure of our data
str(climate_data)

# Look at the first few rows
head(climate_data, 10)


In [ ]:
# Get basic information about our dataset
cat("Number of documents:", nrow(climate_data), "\n")
cat("Number of unique labels:", length(unique(climate_data$label)), "\n")
cat("Label distribution:")
# Convert labels to factor with descriptive levels
climate_data$label <- factor(climate_data$label,
                            levels = 0:4,
                            labels = c("General climate discussion",
                                     "Politics and policy",
                                     "Ocean/Water",
                                     "Agriculture/forestry",
                                     "Climate disasters and impacts"))

table(climate_data$label)


### Initial Data Cleaning

Before we can analyze the text, we need to clean it. Common cleaning steps include:
- Removing URLs and special characters
- Converting to lowercase
- Removing extra whitespace
- Handling missing values


In [ ]:
# Clean the text data
climate_clean <- climate_data %>%
  # Remove rows with missing text
  filter(!is.na(text)) %>%
  # Convert to lowercase
  mutate(text = tolower(text)) %>%
  # Remove URLs (simple pattern)
  mutate(text = str_replace_all(text, "https?://\\S+", "")) %>%
  # Remove Twitter handles
  mutate(text = str_replace_all(text, "@\\w+", "")) %>%
  # Remove hashtags but keep the words
  mutate(text = str_replace_all(text, "#", "")) %>%
  # Remove punctuation
  mutate(text = str_replace_all(text, "[[:punct:]]", " ")) %>%
  # Remove extra whitespace
  mutate(text = str_squish(text)) %>%
  # Remove rows that are now empty
  filter(text != "")

# Check the results
cat("Original number of documents:", nrow(climate_data), "\n")
cat("After cleaning:", nrow(climate_clean), "\n")

# Look at some examples of cleaned text
head(climate_clean$text, 5)


## Part 2: Tokenization and Stopword Removal

### What is Tokenization?

Tokenization is the process of breaking text into individual words (tokens). This is a crucial step in text analysis because it allows us to:
- Count word frequencies
- Analyze word patterns
- Apply statistical methods

### Stopwords

Stopwords are common words that usually don't carry much meaning (e.g., "the", "and", "is"). Removing them helps focus on the meaningful content.


In [ ]:
# Tokenize the text into individual words
climate_tokens <- climate_clean %>%
  # Add row numbers for tracking
  mutate(document_id = row_number()) %>%
  # Tokenize into words
  unnest_tokens(word, text) %>%
  # Remove stopwords
  anti_join(stop_words, by = "word") %>%
  # Remove numbers
  filter(!str_detect(word, "^\\d+")) %>%
  # Remove single characters (except 'a' and 'i')
  filter(nchar(word) > 1)

# Examine the results
head(climate_tokens, 20)


In [ ]:
# Get basic statistics about our tokenized data
cat("Total number of tokens:", nrow(climate_tokens), "\n")
cat("Number of unique words:", length(unique(climate_tokens$word)), "\n")
cat("Average words per document:", round(nrow(climate_tokens) / length(unique(climate_tokens$document_id)), 1), "\n")


## Part 3: Word Frequency Analysis

Now that we have clean, tokenized data, let's analyze word frequencies to understand what topics are most prominent in our climate change discussions.


In [ ]:
# Calculate word frequencies
word_freq <- climate_tokens %>%
  count(word, sort = TRUE) %>%
  mutate(percentage = n / sum(n) * 100)

# Display the top 20 most frequent words
word_freq %>%
  head(20) %>%
  kable(col.names = c("Word", "Count", "Percentage"))


In [ ]:
# Remove ampersand artifact and the terms that the dataset was generated from
climate_tokens <- climate_tokens %>%
  filter(!word %in% c("amp", "climate", "change", "climatechange"))  # Remove ampersand artifacts and climate-related terms

# Recalculate word frequencies after cleaning
word_freq <- climate_tokens %>%
  count(word, sort = TRUE) %>%
  mutate(percentage = n / sum(n) * 100)

# Display updated top 20 most frequent words
word_freq %>%
  head(20) %>%
  kable(col.names = c("Word", "Count", "Percentage"))


In [ ]:
# Create a bar plot of the top 15 most frequent words
word_freq %>%
  head(15) %>%
  ggplot(aes(x = reorder(word, n), y = n)) +
  geom_col(fill = "steelblue", alpha = 0.8) +
  coord_flip() +
  labs(title = "Most Frequent Words in Climate Change Discussions",
       x = "Word",
       y = "Frequency") +
  theme_minimal() +
  theme(axis.text.y = element_text(size = 10))


### Word Cloud Visualization

Word clouds provide an intuitive visual representation of word frequencies, where the size of each word corresponds to its frequency.


In [ ]:
# Prepare data for word cloud (top 100 words)
wordcloud_data <- word_freq %>%
  head(100) %>%
  select(word, n)

# Create word cloud
wordcloud2(wordcloud_data,
          size = 0.8,
          color = "random-dark",
          backgroundColor = "white")


## Part 4: Dictionary-Based Methods

### Sentiment Analysis

Sentiment analysis helps us understand the emotional tone of text. We'll use the Bing lexicon, which categorizes words as positive or negative.


In [ ]:
# Get sentiment lexicons
bing_sentiment <- get_sentiments("bing")
# Display some entries from the Bing sentiment lexicon
head(bing_sentiment, 10)


In [ ]:
# Perform sentiment analysis
climate_sentiment <- climate_tokens %>%
  inner_join(bing_sentiment, by = "word") %>%
  count(document_id, sentiment) %>%
  spread(sentiment, n, fill = 0) %>%
  mutate(sentiment_score = positive - negative)

# Add sentiment back to original documents
climate_with_sentiment <- climate_clean %>%
  mutate(document_id = row_number()) %>%
  left_join(climate_sentiment, by = "document_id") %>%
  replace_na(list(positive = 0, negative = 0, sentiment_score = 0))

# Find the document with the highest combined positive + negative sentiment words
most_emotional <- climate_with_sentiment %>%
  mutate(total_sentiment_words = positive + negative) %>%
  arrange(desc(total_sentiment_words)) %>%
  slice_head(n = 1)

# Get the tokens for this document with sentiment
emotional_doc_tokens <- climate_tokens %>%
  filter(document_id == most_emotional$document_id) %>%
  left_join(bing_sentiment, by = "word") %>%
  mutate(sentiment = replace_na(sentiment, "neutral"))

# Create colored text output
cat("Document with Most Emotional Content (", most_emotional$total_sentiment_words, " sentiment words):\n")
cat("Positive words:", most_emotional$positive, "| Negative words:", most_emotional$negative, "\n")
cat("Overall sentiment score:", most_emotional$sentiment_score, "\n\n")

# Print the text with colored sentiment words
words_with_sentiment <- emotional_doc_tokens %>%
  mutate(colored_word = case_when(
    sentiment == "positive" ~ paste0("\033[32m", word, "\033[0m"),  # Green for positive
    sentiment == "negative" ~ paste0("\033[31m", word, "\033[0m"),  # Red for negative
    TRUE ~ word  # No color for neutral
  ))

# Combine words back into text
colored_text <- paste(words_with_sentiment$colored_word, collapse = " ")
cat(colored_text, "\n\n")

# Print the original document text
original_text <- climate_clean %>%
  filter(row_number() == most_emotional$document_id) %>%
  pull(text)

cat("Original Document Text:\n")
cat(str_wrap(original_text, width = 80), "\n\n")




# Display summary statistics
summary(climate_with_sentiment$sentiment_score)


In [ ]:
# Create a histogram of sentiment scores
ggplot(climate_with_sentiment, aes(x = sentiment_score)) +
  geom_histogram(bins = 30, fill = "steelblue", alpha = 0.7) +
  geom_vline(xintercept = 0, color = "red", linetype = "dashed") +
  labs(title = "Distribution of Sentiment Scores in Climate Change Discussions",
       x = "Sentiment Score (Positive - Negative)",
       y = "Frequency") +
  theme_minimal()


In [ ]:
# Find the most positive and negative tweets
top_positive <- climate_with_sentiment %>%
  arrange(desc(sentiment_score)) %>%
  slice_head(n = 10) %>%
  select(text, sentiment_score, positive, negative)

top_negative <- climate_with_sentiment %>%
  arrange(sentiment_score) %>%
  slice_head(n = 10) %>%
  select(text, sentiment_score, positive, negative)

# Display most positive tweets
cat("Top 10 Most Positive Tweets:\n")
cat("=" %>% rep(50) %>% paste(collapse = ""), "\n\n")
for(i in 1:nrow(top_positive)) {
  cat("Rank", i, "(Score:", top_positive$sentiment_score[i], ")\n")
  cat(str_wrap(top_positive$text[i], width = 80), "\n\n")
}

cat("\nTop 10 Most Negative Tweets:\n")
cat("=" %>% rep(50) %>% paste(collapse = ""), "\n\n")
for(i in 1:nrow(top_negative)) {
  cat("Rank", i, "(Score:", top_negative$sentiment_score[i], ")\n")
  cat(str_wrap(top_negative$text[i], width = 80), "\n\n")
}


### Environment-Specific Dictionary Analysis

Let's create a custom dictionary of environment-related terms to analyze how different environmental topics are discussed.


In [ ]:
# Create environment-related dictionaries
environment_dict <- list(
  climate_terms = c("climate", "warming", "temperature", "emissions", "carbon", "greenhouse"),
  action_terms = c("action", "policy", "agreement", "target", "reduction", "mitigation"),
  impact_terms = c("impact", "effect", "damage", "disaster", "crisis", "emergency"),
  solution_terms = c("solution", "renewable", "solar", "wind", "energy", "technology")
)

# Analyze usage of environment-related terms
env_analysis <- climate_tokens %>%
  mutate(
    climate_related = word %in% environment_dict$climate_terms,
    action_related = word %in% environment_dict$action_terms,
    impact_related = word %in% environment_dict$impact_terms,
    solution_related = word %in% environment_dict$solution_terms
  ) %>%
  group_by(document_id) %>%
  summarise(
    climate_count = sum(climate_related),
    action_count = sum(action_related),
    impact_count = sum(impact_related),
    solution_count = sum(solution_related)
  )

# Display summary
env_analysis %>%
  summarise(
    total_climate = sum(climate_count),
    total_action = sum(action_count),
    total_impact = sum(impact_count),
    total_solution = sum(solution_count)
  ) %>%
  gather(category, count) %>%
  kable( col.names = c("Category", "Total Count"))


In [ ]:
# Visualize environment-related term usage
env_analysis %>%
  summarise(
    climate = sum(climate_count),
    action = sum(action_count),
    impact = sum(impact_count),
    solution = sum(solution_count)
  ) %>%
  gather(category, count) %>%
  ggplot(aes(x = reorder(category, count), y = count)) +
  geom_col(fill = "darkgreen", alpha = 0.8) +
  coord_flip() +
  labs(title = "Usage of Environment-Related Terms",
       x = "Category",
       y = "Count") +
  theme_minimal()


## Exercise: analyze sentiment across the documents in our topics
Which topic do you think will have the most positive sentiment? The most negative?

In [ ]:
# Combine environment analysis with sentiment analysis
env_sentiment_analysis <- env_analysis %>%
  left_join(climate_with_sentiment %>%
        select(document_id, sentiment_score) %>%
        group_by(document_id) %>%
        summarise(avg_sentiment = mean(sentiment_score, na.rm = TRUE)),
    by = "document_id")

# Create separate datasets for each category based on term presence
# YOUR CODE HERE

# Calculate sentiment for each category
# YOUR CODE HERE


# Display summary of environment terms and sentiment
category_sentiment %>%
  kable(col.names = c("Category", "Document Count", "Average Sentiment"))


## Part 5: Advanced Visualizations

### Topic Analysis by Label

Let's analyze how different topics (labels) use different words and have different sentiment patterns.


In [ ]:
# Add labels back to our sentiment analysis
climate_topic_sentiment <- climate_with_sentiment %>%
  select(document_id, label, sentiment_score) %>%
  filter(!is.na(label))

# Analyze sentiment by topic
topic_sentiment_summary <- climate_topic_sentiment %>%
  group_by(label) %>%
  summarise(
    mean_sentiment = mean(sentiment_score, na.rm = TRUE),
    median_sentiment = median(sentiment_score, na.rm = TRUE),
    count = n()
  )

# Display results
topic_sentiment_summary %>%
  kable(
        col.names = c("Topic Label", "Mean Sentiment", "Median Sentiment", "Document Count"))


In [ ]:
# Visualize sentiment by topic
ggplot(climate_topic_sentiment, aes(x = factor(label), y = sentiment_score)) +
  geom_violin(fill = "lightblue", alpha = 0.7) +
  geom_hline(yintercept = 0, color = "red", linetype = "dashed") +
  labs(title = "Sentiment Distribution by Topic",
       x = "Topic Label",
       y = "Sentiment Score") +
  theme_minimal()


## Summary

In this lesson, we've covered the fundamental techniques for analyzing environmental text data:

1. **Data Loading and Cleaning**: We loaded climate-related text data and performed essential cleaning steps
2. **Tokenization and Preprocessing**: We broke text into words and removed stopwords
3. **Word Frequency Analysis**: We identified the most common terms in climate discussions
4. **Dictionary-Based Methods**: We applied sentiment analysis and environment-specific dictionaries
5. **Visualization**: We created word clouds, frequency plots, and topic-specific analyses

These techniques provide a foundation for more advanced text analysis in environmental research, policy analysis, and social media monitoring.
